In [36]:
import numpy as np
import pandas as pd
from numpy.random import default_rng

# ==============================
# Global constants
# ==============================

OMEGA = 2 * np.pi / 365.0  # annual frequency for sin/cos seasonality


# ==============================
# Mean temperature model (eq. 3.9)
# ==============================

def _fit_mean_model(df_region: pd.DataFrame) -> dict:
    """
    Fit T^m_t = a1 + a2 t + a3 sin(ω t) + a4 cos(ω t)
    as in eq. (3.9) of Alaton et al. (2002).

    df_region must have columns: "date", "daily_avg_temperature".
    """
    df = df_region.sort_values("date").dropna(subset=["daily_avg_temperature"]).copy()
    if df.empty:
        raise ValueError("No data for region in _fit_mean_model")

    t0 = df["date"].iloc[0]
    t_idx = (df["date"] - t0).dt.days.to_numpy(dtype=float)
    T = df["daily_avg_temperature"].to_numpy(dtype=float)

    X_design = np.column_stack([
        np.ones_like(t_idx),
        t_idx,
        np.sin(OMEGA * t_idx),
        np.cos(OMEGA * t_idx),
    ])
    beta, *_ = np.linalg.lstsq(X_design, T, rcond=None)
    a1, a2, a3, a4 = beta

    return {
        "t0_date": t0,
        "a1": float(a1),
        "a2": float(a2),
        "a3": float(a3),
        "a4": float(a4),
    }


def _mean_Tm(mean_params: dict, dates: pd.Series | pd.DatetimeIndex):
    """
    Compute T^m_t for a vector of dates using:
        T^m_t = a1 + a2 t + a3 sin(ω t) + a4 cos(ω t),
    where t is days since t0_date.
    """
    t0 = mean_params["t0_date"]
    dates = pd.to_datetime(dates)
    delta = dates - t0

    if isinstance(delta, pd.Series):
        t_idx = delta.dt.days.to_numpy(dtype=float)
    else:  # DatetimeIndex
        t_idx = delta.days.astype(float)

    a1 = mean_params["a1"]
    a2 = mean_params["a2"]
    a3 = mean_params["a3"]
    a4 = mean_params["a4"]

    Tm = a1 + a2 * t_idx + a3 * np.sin(OMEGA * t_idx) + a4 * np.cos(OMEGA * t_idx)
    return Tm, t_idx




In [37]:
# ==============================
# Initial AR(1) estimate for a
# ==============================

def _initial_a_ar1(T: np.ndarray, Tm: np.ndarray) -> float:
    """
    Simple AR(1)-based estimate of mean reversion speed a as a starting value.
    We apply AR(1) to deseasonalised series X_t = T_t - T^m_t.
    """
    X = T - Tm
    X_t = X[:-1]
    X_next = X[1:]
    denom = np.dot(X_t, X_t)
    if denom <= 0:
        return 0.1  # safe fallback

    phi = np.dot(X_t, X_next) / denom
    phi = float(np.clip(phi, 1e-6, 0.999))  # keep in (0,1) for mean reversion
    a = -np.log(phi)
    return a



In [38]:
# ==============================
# Monthly volatilities (two estimators)
# ==============================

def _estimate_monthly_vols(df_region: pd.DataFrame,
                           mean_params: dict,
                           a_hat0: float) -> np.ndarray:
    """
    Implement Alaton's two volatility estimators per calendar month:
      - Estimator 1: quadratic variation of increments (eq. 3.16)
      - Estimator 2: regression-based residuals (eqs. 3.17–3.19)
    Then take their mean.

    Returns vol_by_month[0..11] corresponding to months 1..12.
    """
    df = df_region.sort_values("date").dropna(subset=["daily_avg_temperature"]).copy()
    dates = df["date"]
    T = df["daily_avg_temperature"].to_numpy(dtype=float)
    Tm, _ = _mean_Tm(mean_params, dates)
    months = dates.dt.month.to_numpy()

    vol1_sq = np.full(12, np.nan)
    vol2_sq = np.full(12, np.nan)

    for m in range(1, 13):
        mask = (months == m)
        idxs = np.where(mask)[0]
        if len(idxs) < 3:
            continue

        # --- Estimator 1: quadratic variation (3.16) ---
        deltas = []
        for k in range(len(idxs) - 1):
            i = idxs[k]
            j = idxs[k + 1]
            # require consecutive calendar days
            if (dates.iloc[j] - dates.iloc[i]).days == 1:
                deltas.append(T[j] - T[i])
        if deltas:
            deltas = np.asarray(deltas, dtype=float)
            N_m = len(idxs)
            vol1_sq[m - 1] = np.sum(deltas**2) / N_m

        # --- Estimator 2: regression-based (3.17)-(3.19) ---
        res_sq_sum = 0.0
        count = 0
        for k in range(1, len(idxs)):
            j = idxs[k]
            j_prev = idxs[k - 1]
            if (dates.iloc[j] - dates.iloc[j_prev]).days != 1:
                continue
            # T~_j = T_j - (T^m_j - T^m_{j-1})
            T_tilde = T[j] - (Tm[j] - Tm[j_prev])
            # residual r_j = T~_j - a_hat0 * T^m_{j-1} - (1 - a_hat0) T_{j-1}
            r = T_tilde - a_hat0 * Tm[j_prev] - (1.0 - a_hat0) * T[j_prev]
            res_sq_sum += r * r
            count += 1

        if count > 2:
            vol2_sq[m - 1] = res_sq_sum / (count - 2)

    # Combine: mean of estimator 1 and 2
    vol_sq = np.nanmean(np.vstack([vol1_sq, vol2_sq]), axis=0)

    # Fallback for months with no reliable estimate
    if np.isnan(vol_sq).any():
        diffs = np.diff(T)
        global_var = np.mean(diffs**2) if len(diffs) > 0 else 1.0
        vol_sq = np.where(np.isnan(vol_sq), global_var, vol_sq)

    vol_by_month = np.sqrt(vol_sq)
    return vol_by_month

In [39]:
# ==============================
# Martingale estimator for a (eq. 3.26–3.27)
# ==============================

def _estimate_a_martingale(df_region: pd.DataFrame,
                           mean_params: dict,
                           vol_by_month: np.ndarray) -> float:
    """
    Martingale estimator for mean reversion speed a:

      a_hat = -log( sum Y_{i-1}(T_i - T^m_i) / sum Y_{i-1}(T_{i-1} - T^m_{i-1}) )
      with Y_{i-1} = (T^m_{i-1} - T_{i-1}) / sigma_{i-1}^2

    sigma_{i-1} is monthly volatility corresponding to month of date_{i-1}.
    """
    df = df_region.sort_values("date").dropna(subset=["daily_avg_temperature"]).copy()
    dates = df["date"]
    T = df["daily_avg_temperature"].to_numpy(dtype=float)
    Tm, _ = _mean_Tm(mean_params, dates)
    months = dates.dt.month.to_numpy()

    num = 0.0
    den = 0.0
    pairs_used = 0

    for i in range(1, len(T)):
        m_prev = months[i - 1]
        sigma_prev = vol_by_month[m_prev - 1]
        if not np.isfinite(sigma_prev) or sigma_prev <= 0:
            continue

        Y = (Tm[i - 1] - T[i - 1]) / (sigma_prev**2)
        num += Y * (T[i] - Tm[i])
        den += Y * (T[i - 1] - Tm[i - 1])
        pairs_used += 1

    if pairs_used == 0 or den == 0.0:
        # Fallback to AR(1)-based a if martingale estimator degenerates
        return _initial_a_ar1(T, Tm)

    phi_weighted = num / den
    # For mean reversion we want 0 < phi < 1
    if phi_weighted <= 0 or phi_weighted >= 1:
        return _initial_a_ar1(T, Tm)

    a_hat = -np.log(phi_weighted)
    return float(a_hat)




In [40]:
# ==============================
# Full Alaton-style calibration per region
# ==============================

def fit_ou_region_alaton(df_region: pd.DataFrame) -> dict:
    """
    Full Alaton-style calibration for ONE region:
      1) Fit mean temperature model (3.9)
      2) Initial AR(1) estimate of a
      3) Monthly vols via (3.16) & (3.19) and their mean
      4) Refined a via martingale estimator (3.26)-(3.27)
      5) Re-estimate vols with refined a

    Returns parameter dict ready for pricing.
    """
    df_region = df_region.sort_values("date").dropna(subset=["daily_avg_temperature"]).copy()
    if df_region.empty:
        raise ValueError("Empty df_region in fit_ou_region_alaton")

    # 1) Mean model
    mean_params = _fit_mean_model(df_region)
    dates = df_region["date"]
    T = df_region["daily_avg_temperature"].to_numpy(dtype=float)
    Tm, _ = _mean_Tm(mean_params, dates)

    # 2) Initial a from AR(1) on deseasonalised series
    a_init = _initial_a_ar1(T, Tm)

    # 3) Monthly vols using a_init
    vol_by_month_init = _estimate_monthly_vols(df_region, mean_params, a_init)

    # 4) Refined a via martingale estimator
    a_hat = _estimate_a_martingale(df_region, mean_params, vol_by_month_init)

    # 5) Re-estimate vols with refined a (closer to paper)
    vol_by_month = _estimate_monthly_vols(df_region, mean_params, a_hat)

    # Discrete-time AR coefficient for daily step
    phi = float(np.exp(-a_hat))

    params = {
        "region_code": df_region["region_code"].iloc[0],
        "t0_date": mean_params["t0_date"],
        "a1": mean_params["a1"],
        "a2": mean_params["a2"],
        "a3": mean_params["a3"],
        "a4": mean_params["a4"],
        "a": a_hat,                 # continuous-time mean reversion speed
        "phi": phi,                 # daily AR(1) coefficient
        "vol_by_month": vol_by_month,  # σ_m for months 1..12
    }
    return params




In [41]:
# ==============================
# Seasonal mean μ_t for pricing
# ==============================

def compute_mu_for_dates(params: dict, dates: pd.DatetimeIndex | pd.Series):
    """
    Seasonal mean μ_t = a1 + a2 t + a3 sin(ω t) + a4 cos(ω t),
    where t is days since t0_date.
    """
    t0 = params["t0_date"]
    dates = pd.to_datetime(dates)
    delta = dates - t0

    if isinstance(delta, pd.Series):
        t_idx = delta.dt.days.to_numpy(dtype=float)
    else:
        t_idx = delta.days.astype(float)

    a1 = params["a1"]
    a2 = params["a2"]
    a3 = params["a3"]
    a4 = params["a4"]

    mu = a1 + a2 * t_idx + a3 * np.sin(OMEGA * t_idx) + a4 * np.cos(OMEGA * t_idx)
    return mu, t_idx





In [42]:
# ==============================
# HDD pricing by Monte Carlo
# ==============================

def price_hdd_contract(
    params: dict,
    df_region: pd.DataFrame,
    start_date: str | pd.Timestamp,
    end_date: str | pd.Timestamp,
    base_temp: float = 18.0,
    notional: float = 1.0,
    strike: float | None = None,     # if None -> futures, else call
    r: float = 0.0,
    n_paths: int = 10000,
    seed: int = 42,
    use_last_obs_as_start: bool = True,
    market_price_of_risk: float = 0.0,  # placeholder; currently not used
) -> dict:
    """
    Price an HDD contract (futures or call) for one region via Monte Carlo.

    - params: dict from fit_ou_region_alaton
    - df_region: full regional dataframe (with 'date', 'daily_avg_temperature')
    - start_date, end_date: datetime or 'YYYY-MM-DD' strings (inclusive)
    - strike: if None, returns only futures price; else also call price
    """
    rng = default_rng(seed)

    # Contract date range
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)
    dates = pd.date_range(start_date, end_date, freq="D")
    n_days = len(dates)

    # Mean μ_t and time index
    mu, _ = compute_mu_for_dates(params, dates)

    # Monthly vol profile
    vol_by_month = params["vol_by_month"]  # length 12
    months = dates.month.to_numpy()
    daily_vol = np.array([vol_by_month[m - 1] for m in months], dtype=float)

    phi = params["phi"]

    # Starting X_0 (deseasonalised component)
    if use_last_obs_as_start:
        df_region = df_region.sort_values("date").dropna(subset=["daily_avg_temperature"]).copy()
        mask = df_region["date"] <= start_date
        if mask.any():
            T0 = df_region.loc[mask, "daily_avg_temperature"].iloc[-1]
            mu0, _ = compute_mu_for_dates(params, pd.DatetimeIndex([start_date]))
            X0 = float(T0 - mu0[0])
        else:
            X0 = 0.0
    else:
        X0 = 0.0

    # Simulate OU paths
    X_paths = np.zeros((n_paths, n_days), dtype=float)
    T_paths = np.zeros_like(X_paths)

    X_paths[:, 0] = X0
    T_paths[:, 0] = mu[0] + X0

    Z = rng.standard_normal((n_paths, n_days - 1))

    for d in range(n_days - 1):
        sigma_d = daily_vol[d]
        eps = sigma_d * Z[:, d]   # innovations
        X_next = phi * X_paths[:, d] + eps
        X_paths[:, d + 1] = X_next
        T_paths[:, d + 1] = mu[d + 1] + X_next

    # HDD index per path
    hdd_daily = np.maximum(base_temp - T_paths, 0.0)
    hdd_index = hdd_daily.sum(axis=1)

    # Futures & option prices
    tau_years = n_days / 365.0
    disc = np.exp(-r * tau_years)
    expected_hdd = float(hdd_index.mean())
    fut_price = float(disc * notional * expected_hdd)

    if strike is None:
        call_price = None
    else:
        payoff_call = np.maximum(hdd_index - strike, 0.0) * notional
        call_price = float(disc * payoff_call.mean())

    return {
        "expected_hdd": expected_hdd,
        "future_price": fut_price,
        "call_price": call_price,
    }



In [44]:
# ==============================
# Main: load data, calibrate, price all regions
# ==============================

if __name__ == "__main__":
    # 1) Load data
    # Adjust the path if needed
    df = pd.read_csv("../EDA/region_avg.csv", parse_dates=["date"])

    # 2) Calibrate OU model for each region (Alaton-style)
    region_params = {}
    for reg in sorted(df["region_code"].unique()):
        df_reg = df[df["region_code"] == reg].copy()
        try:
            params = fit_ou_region_alaton(df_reg)
            region_params[reg] = params
        except Exception as e:
            print(f"Calibration failed for region {reg}: {e}")

    print(f"Calibrated {len(region_params)} regions.\n")

    # 3) Price a sample HDD January contract for each region
    #    (example: Jan 2010, strike 400, base 18°C, notional 1, r=0)
    results = []
    for reg, params in region_params.items():
        df_reg = df[df["region_code"] == reg].copy()
        res = price_hdd_contract(
            params=params,
            df_region=df_reg,
            start_date="2010-01-01",
            end_date="2010-01-31",
            base_temp=18.0,
            notional=1.0,
            strike=400.0,
            r=0.0,
            n_paths=5000,
            seed=123 + int(reg),  # different seed per region
        )
        results.append({
            "region_code": reg,
            "expected_hdd": res["expected_hdd"],
            "future_price": res["future_price"],
            "call_price": res["call_price"],
        })

    df_results = pd.DataFrame(results).sort_values("region_code")
    print(df_results)

Calibrated 13 regions.

    region_code  expected_hdd  future_price  call_price
0            11    447.536933    447.536933   52.992128
1            24    448.068006    448.068006   53.933576
2            27    470.802655    470.802655   74.239368
3            28    416.913031    416.913031   27.981661
4            32    453.246991    453.246991   56.969045
5            44    482.849377    482.849377   85.292524
6            52    397.836740    397.836740   19.579349
7            53    365.622489    365.622489    4.665625
8            75    378.831806    378.831806   12.345251
9            76    390.592051    390.592051   16.037379
10           84    454.531908    454.531908   59.237788
11           93    359.562970    359.562970    5.674869
12           94    263.667000    263.667000    0.000000
